In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# %cd /content/drive/MyDrive

In [3]:
# !unzip -q dataset.zip

# zh-vi-MT-NLLB: Chinese → Vietnamese Machine Translation
**Model:** `facebook/nllb-200-distilled-600M` + LoRA fine-tuning  
**Tokenizer:** SentencePiece (built-in NLLB)  
**Metrics:** SacreBLEU + COMET  
**Framework:** HuggingFace Transformers + PEFT  
**Hardware:** Google Colab T4 GPU

## 1. Install Dependencies

In [4]:
# !pip uninstall -y transformers peft accelerate trl

In [5]:
# !pip install -q \
# transformers==4.41.2 \
# peft==0.10.0 \
# accelerate==0.29.3 \
# datasets \
# evaluate \
# sacrebleu \
# unbabel-comet \
# sentencepiece

## 2. Imports & Configuration

In [6]:
import os
import random
import math
import json
import zipfile
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from IPython.display import clear_output, display
from dataclasses import dataclass, field
from typing import Optional, List, Dict

import torch
from torch.utils.data import Dataset

import transformers
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    TrainerCallback,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
import sacrebleu as sb
from comet import download_model, load_from_checkpoint

warnings.filterwarnings("ignore")
matplotlib.use("Agg")  # Non-interactive backend safe for Colab

print(f"PyTorch       : {torch.__version__}")
print(f"Transformers  : {transformers.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU           : {torch.cuda.get_device_name(0)}")

PyTorch       : 2.10.0+cu128
Transformers  : 4.41.2
CUDA available: True
GPU           : Tesla T4


In [7]:
# ─── Paths ────────────────────────────────────────────────────────────────────
TRAIN_SRC      = "/content/drive/MyDrive/dataset/train/train.zh"
TRAIN_TGT      = "/content/drive/MyDrive/dataset/train/train.vi"
TEST_SRC       = "/content/drive/MyDrive/dataset/public_test/public_test.zh"
PRIVATE_SRC    = "/content/drive/MyDrive/dataset/private_test/private_test.zh"
SAVE_DIR       = "./nllb_checkpoints"
BEST_MODEL_DIR = os.path.join(SAVE_DIR, "best_model")
LOG_FILE       = os.path.join(SAVE_DIR, "training_log.json")

# ─── NLLB language tags ───────────────────────────────────────────────────────
SRC_LANG = "zho_Hans"   # Simplified Chinese
TGT_LANG = "vie_Latn"   # Vietnamese

# ─── Model ────────────────────────────────────────────────────────────────────
MODEL_NAME = "facebook/nllb-200-distilled-600M"

# ─── Hyperparameters ──────────────────────────────────────────────────────────
SEED              = 42
MAX_LEN           = 64          # token length per sequence
TRAIN_RATIO       = 0.9          # 90% train / 10% val
BATCH_SIZE        = 16           # per-device; T4 ~15GB, safe with gradient_checkpointing
GRAD_ACCUM        = 2            # effective batch = 32
NUM_EPOCHS        = 10
LR                = 3e-4         # LoRA learning rate
WARMUP_RATIO      = 0.05
WEIGHT_DECAY      = 0.01
FP16              = torch.cuda.is_available()
EVAL_STEPS        = 200          # evaluate every N steps
SAVE_TOTAL_LIMIT  = 3
EARLY_STOPPING_PATIENCE = 2      # stop if val_loss doesn't improve for 2 evals

# ─── LoRA ─────────────────────────────────────────────────────────────────────
LORA_R          = 16
LORA_ALPHA      = 32
LORA_DROPOUT    = 0.05
LORA_TARGET     = ["q_proj", "v_proj", "k_proj", "out_proj"]  # attention layers

# ─── COMET model ─────────────────────────────────────────────────────────────
COMET_MODEL     = "Unbabel/wmt22-comet-da"   # reference-based COMET
COMET_BATCH     = 64

# ─── Misc ─────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(SAVE_DIR, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Configuration ready.")
print(f"  Device        : {DEVICE}")
print(f"  Model         : {MODEL_NAME}")
print(f"  LoRA r/alpha  : {LORA_R}/{LORA_ALPHA}")
print(f"  Batch size    : {BATCH_SIZE} × {GRAD_ACCUM} accum = {BATCH_SIZE*GRAD_ACCUM}")
print(f"  FP16          : {FP16}")

Configuration ready.
  Device        : cuda
  Model         : facebook/nllb-200-distilled-600M
  LoRA r/alpha  : 16/32
  Batch size    : 16 × 2 accum = 32
  FP16          : True


## 3. Data Loading & Split

In [9]:
def read_lines(path: str) -> List[str]:
    with open(path, "r", encoding="utf-8") as f:
        return [l.strip() for l in f if l.strip()]

train_src_all = read_lines(TRAIN_SRC)
train_tgt_all = read_lines(TRAIN_TGT)
test_src      = read_lines(TEST_SRC)
private_src   = read_lines(PRIVATE_SRC)

# Shuffle then split
pairs = list(zip(train_src_all, train_tgt_all))
random.shuffle(pairs)
n_train = int(len(pairs) * TRAIN_RATIO)
train_pairs = pairs[:n_train]
val_pairs   = pairs[n_train:]

train_src_split, train_tgt_split = zip(*train_pairs)
val_src_split,   val_tgt_split   = zip(*val_pairs)

print(f"Total pairs    : {len(pairs):,}")
print(f"Train pairs    : {len(train_pairs):,}")
print(f"Val pairs      : {len(val_pairs):,}")
print(f"Public test    : {len(test_src):,}")
print(f"\nExample ZH: {train_src_split[0]}")
print(f"Example VI: {train_tgt_split[0]}")

Total pairs    : 32,061
Train pairs    : 28,854
Val pairs      : 3,207
Public test    : 1,781

Example ZH: 这个 巴士 去 联合 广场 的 度假 旅馆 吗 ？
Example VI: Xe_buýt này có đi đến Quảng_trường Holiday_Inn_Union không ?


## 4. Tokenizer (SentencePiece via NLLB)

In [10]:
# NLLB uses its own SentencePiece tokenizer — no need to train a separate one
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    src_lang=SRC_LANG,
    tgt_lang=TGT_LANG,
)

print(f"Tokenizer type : {type(tokenizer).__name__}")
print(f"Vocab size     : {tokenizer.vocab_size:,}")
print(f"Model max len  : {tokenizer.model_max_length}")

# Quick sanity check
sample = tokenizer(train_src_split[0], return_tensors="pt")
print(f"\nSample input_ids shape: {sample['input_ids'].shape}")
print(f"Decoded back: {tokenizer.decode(sample['input_ids'][0], skip_special_tokens=True)}")

Tokenizer type : NllbTokenizerFast
Vocab size     : 256,204
Model max len  : 1024

Sample input_ids shape: torch.Size([1, 21])
Decoded back: 这个 巴士 去 联合 广场 的 度假 旅馆 吗?


## 5. Dataset

In [11]:
class NLLBTranslationDataset(Dataset):
    """Tokenized zh→vi translation dataset for NLLB."""

    def __init__(self, src_sentences, tgt_sentences, tokenizer, max_len=MAX_LEN):
        self.src = list(src_sentences)
        self.tgt = list(tgt_sentences)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        # Encode source (Chinese)
        tokenizer.src_lang = SRC_LANG
        model_inputs = tokenizer(
            self.src[idx],
            max_length=self.max_len,
            truncation=True,
            padding=False,
        )
        # Encode target (Vietnamese)
        with tokenizer.as_target_tokenizer():
            labels = tokenizer(
                self.tgt[idx],
                max_length=self.max_len,
                truncation=True,
                padding=False,
            )
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs


train_dataset = NLLBTranslationDataset(train_src_split, train_tgt_split, tokenizer)
val_dataset   = NLLBTranslationDataset(val_src_split,   val_tgt_split,   tokenizer)

print(f"Train dataset : {len(train_dataset):,} samples")
print(f"Val dataset   : {len(val_dataset):,} samples")
print(f"\nSample keys   : {list(train_dataset[0].keys())}")

Train dataset : 28,854 samples
Val dataset   : 3,207 samples

Sample keys   : ['input_ids', 'attention_mask', 'labels']


## 6. Model + LoRA

In [22]:
# =========================================================
# LOAD MODEL + APPLY LORA (STABLE VERSION)
# =========================================================

import torch
from transformers import AutoModelForSeq2SeqLM
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
)

# ---------------------------------------------------------
# LOAD BASE MODEL
# ---------------------------------------------------------

base_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,   # Trainer sẽ tự dùng AMP fp16
)

# IMPORTANT
base_model.config.use_cache = False

# ---------------------------------------------------------
# LORA CONFIG
# ---------------------------------------------------------

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    inference_mode=False,

    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,

    bias="none",

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "out_proj",
    ],
)

# ---------------------------------------------------------
# APPLY PEFT
# ---------------------------------------------------------

model = get_peft_model(base_model, lora_config)

# IMPORTANT
model.train()

# IMPORTANT
model.enable_input_require_grads()

# IMPORTANT
model.config.use_cache = False

# ---------------------------------------------------------
# FORCE ONLY LORA TRAINABLE
# ---------------------------------------------------------

for param in model.parameters():
    param.requires_grad = False

for name, param in model.named_parameters():
    if "lora_" in name:
        param.requires_grad = True

        # IMPORTANT FOR FP16 TRAINING
        param.data = param.data.float()

# ---------------------------------------------------------
# VERIFY
# ---------------------------------------------------------

model.print_trainable_parameters()

print("\nFirst trainable layer:")

for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)
        print("dtype:", param.dtype)
        break

trainable params: 4,718,592 || all params: 619,792,384 || trainable%: 0.7613181642451418

First trainable layer:
base_model.model.model.encoder.layers.0.self_attn.k_proj.lora_A.default.weight
dtype: torch.float32


In [23]:
model.print_trainable_parameters()

trainable params: 4,718,592 || all params: 619,792,384 || trainable%: 0.7613181642451418


## 7. Metrics: SacreBLEU + COMET

In [24]:
# Download COMET model once
comet_model_path = download_model(COMET_MODEL)
comet_metric     = load_from_checkpoint(comet_model_path)
print("COMET model loaded.")


def decode_predictions(pred_ids, label_ids, tokenizer):
    """Convert token ids → text, replacing -100 in labels."""
    pred_ids  = np.where(pred_ids  != -100, pred_ids,  tokenizer.pad_token_id)
    label_ids = np.where(label_ids != -100, label_ids, tokenizer.pad_token_id)
    preds  = tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    labels = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    return preds, labels


def compute_metrics(eval_preds):
    """Called by Trainer after each eval step."""
    pred_ids, label_ids = eval_preds

    # For seq2seq the Trainer may return logits; take argmax
    if isinstance(pred_ids, tuple):
        pred_ids = pred_ids[0]
    if pred_ids.ndim == 3:
        pred_ids = pred_ids.argmax(-1)

    preds, refs = decode_predictions(pred_ids, label_ids, tokenizer)

    # ── SacreBLEU ────────────────────────────────────────────────────────────
    bleu = sb.corpus_bleu(preds, [refs]).score

    # ── COMET ────────────────────────────────────────────────────────────────
    # Build source list from val dataset (aligned)
    # We use a global _val_sources set during training callback
    try:
        comet_data = [
            {"src": s, "mt": h, "ref": r}
            for s, h, r in zip(_val_sources_cache, preds, refs)
        ]
        comet_scores = comet_metric.predict(
            comet_data,
            batch_size=COMET_BATCH,
            gpus=1 if torch.cuda.is_available() else 0,
            progress_bar=False,
        )
        comet_score = float(np.mean(comet_scores.scores)) * 100
    except Exception:
        comet_score = 0.0

    return {"sacrebleu": round(bleu, 4), "comet": round(comet_score, 4)}


# Cache val sources so compute_metrics can access them
_val_sources_cache = list(val_src_split)
print("Metrics helper ready.")

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.1. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../root/.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`


COMET model loaded.
Metrics helper ready.


## 8. Real-Time Visualization Callback

In [25]:
class LivePlotCallback(TrainerCallback):
    """
    Logs train loss, val loss, SacreBLEU, and COMET after every
    evaluation step and renders live charts in the notebook.
    """

    def __init__(self, log_path: str = LOG_FILE):
        self.log_path = log_path
        self.history: Dict[str, List] = {
            "step":        [],
            "train_loss":  [],
            "val_loss":    [],
            "sacrebleu":   [],
            "comet":       [],
        }
        # running train loss accumulator
        self._train_loss_sum   = 0.0
        self._train_loss_steps = 0

    # ── Track training loss between evals ─────────────────────────────────
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            self._train_loss_sum   += logs["loss"]
            self._train_loss_steps += 1

    # ── Record & plot after each eval ─────────────────────────────────────
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics is None:
            return

        avg_train = (
            self._train_loss_sum / self._train_loss_steps
            if self._train_loss_steps > 0 else float("nan")
        )
        self._train_loss_sum   = 0.0
        self._train_loss_steps = 0

        self.history["step"].append(state.global_step)
        self.history["train_loss"].append(avg_train)
        self.history["val_loss"].append(metrics.get("eval_loss", float("nan")))
        self.history["sacrebleu"].append(metrics.get("eval_sacrebleu", 0.0))
        self.history["comet"].append(metrics.get("eval_comet", 0.0))

        # Persist log
        with open(self.log_path, "w") as f:
            json.dump(self.history, f, indent=2)

        self._render(state)

    def _render(self, state):
        h = self.history
        if len(h["step"]) < 1:
            return

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        fig.suptitle(
            f"Training Progress  |  Step {state.global_step}  |  "
            f"Best val_loss = {min(x for x in h['val_loss'] if not math.isnan(x)):.4f}",
            fontsize=13, fontweight="bold"
        )

        steps = h["step"]

        # ── Loss ──────────────────────────────────────────────────────────
        ax = axes[0]
        ax.plot(steps, h["train_loss"], "o-", color="steelblue",   label="Train Loss", linewidth=2)
        ax.plot(steps, h["val_loss"],   "s-", color="tomato",      label="Val Loss",   linewidth=2)
        ax.set_title("Loss"); ax.set_xlabel("Step"); ax.set_ylabel("CE Loss")
        ax.legend(); ax.grid(True, alpha=0.3)

        # ── SacreBLEU ─────────────────────────────────────────────────────
        ax = axes[1]
        ax.plot(steps, h["sacrebleu"], "D-", color="mediumseagreen", linewidth=2)
        ax.set_title("SacreBLEU"); ax.set_xlabel("Step"); ax.set_ylabel("BLEU")
        ax.grid(True, alpha=0.3)

        # ── COMET ──────────────────────────────────────────────────────────
        ax = axes[2]
        ax.plot(steps, h["comet"], "P-", color="darkorange", linewidth=2)
        ax.set_title("COMET Score"); ax.set_xlabel("Step"); ax.set_ylabel("COMET")
        ax.grid(True, alpha=0.3)

        plt.tight_layout()
        fig_path = os.path.join(SAVE_DIR, "training_curves.png")
        plt.savefig(fig_path, dpi=100, bbox_inches="tight")
        clear_output(wait=True)
        display(fig)
        plt.close(fig)


live_plot_cb = LivePlotCallback()
print("LivePlotCallback ready.")

LivePlotCallback ready.


## 9. Trainer Configuration

In [26]:
# =========================================================
# DATA COLLATOR
# =========================================================

from transformers import (
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8 if FP16 else None,
)

# =========================================================
# TRAINING ARGS (STABLE FOR NLLB + LORA)
# =========================================================

training_args = Seq2SeqTrainingArguments(
    output_dir=SAVE_DIR,

    # -----------------------------------------------------
    # TRAINING
    # -----------------------------------------------------

    num_train_epochs=NUM_EPOCHS,

    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,

    gradient_accumulation_steps=GRAD_ACCUM,

    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,

    warmup_ratio=WARMUP_RATIO,

    lr_scheduler_type="cosine",

    fp16=FP16,

    # VERY IMPORTANT
    gradient_checkpointing=False,

    # -----------------------------------------------------
    # EVAL & SAVE
    # -----------------------------------------------------

    evaluation_strategy="steps",
    eval_steps=EVAL_STEPS,

    save_strategy="steps",
    save_steps=EVAL_STEPS,

    save_total_limit=SAVE_TOTAL_LIMIT,

    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # -----------------------------------------------------
    # GENERATION
    # -----------------------------------------------------

    predict_with_generate=True,

    generation_max_length=MAX_LEN,
    generation_num_beams=2,

    # -----------------------------------------------------
    # LOGGING
    # -----------------------------------------------------

    logging_dir=os.path.join(SAVE_DIR, "logs"),

    logging_steps=50,

    report_to="none",

    # -----------------------------------------------------
    # IMPORTANT
    # -----------------------------------------------------

    remove_unused_columns=False,

    dataloader_num_workers=2,

    seed=SEED,
)

# =========================================================
# FORCE TARGET LANGUAGE
# =========================================================

forced_bos_token_id = tokenizer.convert_tokens_to_ids(TGT_LANG)

model.config.forced_bos_token_id = forced_bos_token_id

# =========================================================
# TRAINER
# =========================================================

trainer = Seq2SeqTrainer(
    model=model,

    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    tokenizer=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE
        ),
        live_plot_cb,
    ],
)

print("Trainer configured.")

print(f"Forced BOS token : {TGT_LANG}")
print(f"Forced BOS ID    : {forced_bos_token_id}")

# =========================================================
# SANITY CHECK BEFORE TRAIN
# =========================================================

batch = next(iter(trainer.get_train_dataloader()))

batch = {
    k: v.to(model.device)
    for k, v in batch.items()
}

outputs = model(**batch)

print("\nSanity check:")
print("Loss                :", outputs.loss.item())
print("requires_grad       :", outputs.loss.requires_grad)
print("grad_fn             :", outputs.loss.grad_fn)

assert outputs.loss.requires_grad, \
    "ERROR: loss graph detached!"

Trainer configured.
Forced BOS token : vie_Latn
Forced BOS ID    : 256193

Sanity check:
Loss                : 4.748457908630371
requires_grad       : True
grad_fn             : <NllLossBackward0 object at 0x793d2bb09ba0>


## 10. Training

In [27]:
batch = next(iter(trainer.get_train_dataloader()))

for k, v in batch.items():
    print(k, v.shape, v.dtype)

input_ids torch.Size([16, 24]) torch.int64
attention_mask torch.Size([16, 24]) torch.int64
labels torch.Size([16, 32]) torch.int64


In [28]:
sample = train_dataset[0]

print(sample.keys())

dict_keys(['input_ids', 'attention_mask', 'labels'])


In [29]:
print(sample["labels"][:20])

[256193, 60029, 248120, 276, 14390, 4357, 1601, 3596, 5877, 2353, 12229, 248120, 480, 5576, 245298, 248120, 227639, 248120, 58146, 2772]


In [30]:
batch = next(iter(trainer.get_train_dataloader()))

batch = {
    k: v.to(model.device)
    for k, v in batch.items()
}

outputs = model(**batch)

print(outputs.loss)
print(outputs.loss.requires_grad)
print(outputs.loss.grad_fn)

tensor(4.6799, device='cuda:0', grad_fn=<NllLossBackward0>)
True


In [31]:
print("Starting training...")
print(f"  Steps per epoch : {len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)}")
print(f"  Total epochs    : {NUM_EPOCHS}  (EarlyStopping patience={EARLY_STOPPING_PATIENCE})")
print()

train_result = trainer.train()

print("\n✅ Training complete.")
print(f"   Steps         : {train_result.global_step}")
print(f"   Train loss    : {train_result.training_loss:.4f}")

Starting training...
  Steps per epoch : 901
  Total epochs    : 10  (EarlyStopping patience=2)



Step,Training Loss,Validation Loss


OutOfMemoryError: CUDA out of memory. Tried to allocate 3.36 GiB. GPU 0 has a total capacity of 14.56 GiB of which 3.34 GiB is free. Including non-PyTorch memory, this process has 11.22 GiB memory in use. Of the allocated memory 9.72 GiB is allocated by PyTorch, and 1.37 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## 11. Save Best Model

In [ ]:
# The Trainer has already loaded the best checkpoint (load_best_model_at_end=True)
# Save LoRA adapter + tokenizer
trainer.save_model(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)

# Also save training metrics
trainer.save_metrics("train", train_result.metrics)
trainer.save_state()

print(f"Best model saved to: {BEST_MODEL_DIR}")
print(f"Best eval_loss     : {trainer.state.best_metric:.4f}")

## 12. Final Evaluation on Validation Set

In [ ]:
eval_results = trainer.evaluate(
    eval_dataset=val_dataset,
    metric_key_prefix="final_eval",
)

print("\n📊 Final Evaluation Results:")
for k, v in eval_results.items():
    print(f"   {k:30s}: {v:.4f}")

## 13. Final Training Curves

In [ ]:
with open(LOG_FILE) as f:
    h = json.load(f)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Training Summary — zh→vi NLLB-600M + LoRA", fontsize=14, fontweight="bold")

steps = h["step"]

axes[0].plot(steps, h["train_loss"], "o-", color="steelblue",    label="Train Loss", linewidth=2)
axes[0].plot(steps, h["val_loss"],   "s-", color="tomato",       label="Val Loss",   linewidth=2)
axes[0].set_title("Loss"); axes[0].set_xlabel("Step"); axes[0].set_ylabel("CE Loss")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(steps, h["sacrebleu"], "D-", color="mediumseagreen", linewidth=2)
axes[1].set_title("SacreBLEU"); axes[1].set_xlabel("Step"); axes[1].set_ylabel("BLEU")
axes[1].grid(True, alpha=0.3)

axes[2].plot(steps, h["comet"], "P-", color="darkorange", linewidth=2)
axes[2].set_title("COMET Score"); axes[2].set_xlabel("Step"); axes[2].set_ylabel("COMET")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "training_summary.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: training_summary.png")

## 14. Inference Helper

In [ ]:
@torch.no_grad()
def translate_batch(
    sentences: List[str],
    model,
    tokenizer,
    src_lang: str = SRC_LANG,
    tgt_lang: str = TGT_LANG,
    max_len: int = MAX_LEN,
    num_beams: int = 4,
    batch_size: int = 32,
) -> List[str]:
    """Batch-translate sentences from src_lang to tgt_lang."""
    model.eval()
    tokenizer.src_lang = src_lang
    forced_bos = tokenizer.lang_code_to_id[tgt_lang]
    outputs = []

    for i in range(0, len(sentences), batch_size):
        batch = sentences[i : i + batch_size]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_len,
        ).to(DEVICE)

        gen_ids = model.generate(
            **inputs,
            forced_bos_token_id=forced_bos,
            num_beams=num_beams,
            max_new_tokens=max_len,
            early_stopping=True,
        )
        decoded = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)
        outputs.extend(decoded)

    return outputs


# Quick demo
demo = ["你好，世界！", "机器翻译是人工智能的重要应用。"]
demo_out = translate_batch(demo, model, tokenizer)
for zh, vi in zip(demo, demo_out):
    print(f"ZH: {zh}")
    print(f"VI: {vi}")
    print()

## 15. Translate Test Sets

In [ ]:
from tqdm.auto import tqdm

def translate_and_save(src_sentences, out_path, desc="Translating"):
    print(f"Translating {len(src_sentences):,} sentences → {out_path}")
    translations = []
    for i in tqdm(range(0, len(src_sentences), 32), desc=desc):
        batch = src_sentences[i : i + 32]
        translations.extend(translate_batch(batch, model, tokenizer))
    df = pd.DataFrame({"tieng_trung": src_sentences, "tieng_viet": translations})
    df.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"Saved {len(df):,} rows → {out_path}")
    return translations


public_preds  = translate_and_save(test_src,    "public_test.csv",  "Public test")
private_preds = translate_and_save(private_src, "private_test.csv", "Private test")

# Show samples
print("\n--- Public test samples ---")
for zh, vi in zip(test_src[:20], public_preds[:20]):
    print(f"  ZH: {zh}")
    print(f"  VI: {vi}")
    print()

## 16. Submission ZIP

In [ ]:
zip_path = "submission.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write("public_test.csv",  arcname="public_test.csv")
    zf.write("private_test.csv", arcname="private_test.csv")

print(f"✅ Submission ZIP: {zip_path}  ({os.path.getsize(zip_path)/1024:.1f} KB)")

## 17. (Optional) Load Best Model from Disk

In [ ]:
# Run this cell independently to reload the saved best model

# from peft import PeftModel
# from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
#
# tokenizer_reload = AutoTokenizer.from_pretrained(BEST_MODEL_DIR)
# base_reload = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
# model_reload = PeftModel.from_pretrained(base_reload, BEST_MODEL_DIR)
# model_reload = model_reload.merge_and_unload()  # merge LoRA → full weights
# model_reload.eval().to(DEVICE)
# print("Best model reloaded & merged.")